# 📥 Azure SQL Advisor — Data Collector

**Notebook 1 of 2** — Connects to Azure SQL Database via JDBC, extracts 42 metrics
(2 incremental + 40 snapshot), and persists partitioned Parquet files to ADLS Gen2.

| Component | Detail |
|---|---|
| **Extraction** | JDBC pushdown via PySpark `spark.read.jdbc()` |
| **Incremental** | Watermark-based (resource_stats, query_store_stats) |
| **Snapshot** | Full extraction each run (42 DMV/catalog queries) |
| **Storage** | ADLS Gen2 Parquet, partitioned by `year/month/day` |
| **Watermark** | JSON file in `_metadata/watermarks.json` |

In [ ]:
# Databricks notebook source
# Create interactive widgets
dbutils.widgets.text("server_name", "", "Azure SQL Server FQDN")
dbutils.widgets.text("database_name", "", "Database Name")
dbutils.widgets.text("secret_scope", "azure-sql-credentials", "Secret Scope")
dbutils.widgets.text("username_key", "sql-username", "Username Secret Key")
dbutils.widgets.text("password_key", "sql-password", "Password Secret Key")
dbutils.widgets.text("storage_account", "", "Storage Account Name")
dbutils.widgets.text("storage_container", "azure-sql-telemetry", "Container Name")

### 📦 Import Configuration

In [ ]:
import sys, os, json
from datetime import datetime, timezone, timedelta

repo_path = os.path.dirname(os.path.abspath(globals().get('__file__', '/Workspace/Repos/azure_sql_advisor')))
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

from config import (
    AdvisorConfig, INCREMENTAL_QUERIES, SNAPSHOT_QUERIES, ALL_METRICS
)

# Read widget values
server_name = dbutils.widgets.get("server_name")
database_name = dbutils.widgets.get("database_name")
secret_scope = dbutils.widgets.get("secret_scope")
username_key = dbutils.widgets.get("username_key")
password_key = dbutils.widgets.get("password_key")
storage_account = dbutils.widgets.get("storage_account")
storage_container = dbutils.widgets.get("storage_container")

config = AdvisorConfig(
    server=server_name,
    database=database_name,
    storage_account_name=storage_account,
    storage_container=storage_container,
)

# Storage paths
base_path = f"abfss://{storage_container}@{storage_account}.dfs.core.windows.net/raw/{server_name}/{database_name}"
watermark_path = f"{base_path}/_metadata/watermarks.json"

print(f"Target:  {server_name}/{database_name}")
print(f"Storage: {base_path}")
print(f"Total metrics to collect: {len(ALL_METRICS)}")

### 🔐 Establish JDBC Connection

In [ ]:
# Build JDBC URL and properties
jdbc_url = f"jdbc:sqlserver://{server_name}:1433;database={database_name};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30"

username = dbutils.secrets.get(scope=secret_scope, key=username_key)
password = dbutils.secrets.get(scope=secret_scope, key=password_key)

jdbc_properties = {
    "user": username,
    "password": password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    "fetchsize": "1000",
}

def read_sql(query):
    """Execute a SQL query against Azure SQL via JDBC pushdown."""
    return spark.read.jdbc(url=jdbc_url, table=f"({query}) AS q", properties=jdbc_properties)

def safe_read_sql(query, name):
    """Execute query with error handling."""
    try:
        df = read_sql(query)
        count = df.count()
        print(f"  ✓ {name}: {count} rows")
        return df
    except Exception as e:
        print(f"  ✗ {name}: FAILED - {str(e)[:200]}")
        return None

# Test connectivity
test_df = read_sql("SELECT DB_NAME() AS db_name, GETUTCDATE() AS utc_now")
print(f"✅ Connected to: {test_df.collect()[0]['db_name']}")

### 📏 Watermark Management (Incremental Extraction)

In [ ]:
from pyspark.sql.functions import col, lit, max as spark_max

def load_watermarks():
    """Load existing watermarks from ADLS Gen2."""
    try:
        wm_text = dbutils.fs.head(watermark_path.replace("abfss://", "dbfs:/").replace(".dfs.core.windows.net", ""), 10000)
        return json.loads(wm_text)
    except Exception:
        print("  No existing watermarks found — starting fresh.")
        return {}

def save_watermarks(wm):
    """Persist watermarks back to ADLS Gen2."""
    wm_json = json.dumps(wm, indent=2, default=str)
    dbutils.fs.put(watermark_path.replace("abfss://", "dbfs:/").replace(".dfs.core.windows.net", ""), wm_json, overwrite=True)
    print(f"  ✓ Watermarks saved ({len(wm)} metrics)")

watermarks = load_watermarks()
print(f"Loaded {len(watermarks)} existing watermarks")

### 🚀 Extract & Persist All Metrics

Runs **2 incremental** + **40 snapshot** queries, writing Parquet to ADLS Gen2.

In [ ]:
from datetime import datetime, timezone

now = datetime.now(timezone.utc)
partition_path = f"year={now.year}/month={now.month:02d}/day={now.day:02d}"
collection_summary = {"success": [], "failed": [], "skipped": []}

# ── INCREMENTAL QUERIES ──
print("=== Incremental Queries ===")
for metric_name, q_spec in INCREMENTAL_QUERIES.items():
    base_sql = q_spec['sql']
    wm_col = q_spec['watermark_column']
    last_wm = watermarks.get(metric_name, {}).get('last_value', None)

    if last_wm:
        where_clause = f"WHERE {wm_col} > '{last_wm}'"
    else:
        where_clause = ""

    query = base_sql.format(where_clause=where_clause)
    df = safe_read_sql(query, metric_name)

    if df is not None and df.count() > 0:
        output_path = f"{base_path}/{metric_name}/{partition_path}"
        df.write.mode("append").parquet(output_path)

        # Update watermark
        new_wm = df.agg(spark_max(col(wm_col))).collect()[0][0]
        watermarks[metric_name] = {
            'last_value': str(new_wm),
            'last_run': now.isoformat(),
            'rows_extracted': df.count()
        }
        collection_summary["success"].append(metric_name)
    elif df is not None:
        collection_summary["skipped"].append(metric_name)
        print(f"  ⏭ {metric_name}: No new rows since watermark")
    else:
        collection_summary["failed"].append(metric_name)

# ── SNAPSHOT QUERIES ──
print("\n=== Snapshot Queries ===")
for metric_name, query_template in SNAPSHOT_QUERIES.items():
    # Substitute config placeholders
    query = query_template.format(
        top_queries_count=config.top_queries_count,
        min_execution_count=config.min_execution_count,
        missing_index_impact_threshold=config.missing_index_impact_threshold,
        min_index_pages=config.min_index_pages,
        fragmentation_reorg_pct=config.fragmentation_reorg_pct,
        sp_top_count=config.sp_top_count,
        sp_min_execution_count=config.sp_min_execution_count,
        wide_table_column_threshold=config.wide_table_column_threshold,
        fk_missing_index_min_rows=config.fk_missing_index_min_rows,
        archival_min_size_mb=config.archival_min_size_mb,
        partition_candidate_min_gb=config.partition_candidate_min_gb,
        stale_stats_days=config.stale_stats_days,
        long_transaction_threshold_seconds=config.long_transaction_threshold_seconds,
    )
    df = safe_read_sql(query, metric_name)

    if df is not None and df.count() > 0:
        output_path = f"{base_path}/{metric_name}/{partition_path}"
        df.write.mode("overwrite").parquet(output_path)
        watermarks[metric_name] = {
            'last_run': now.isoformat(),
            'rows_extracted': df.count()
        }
        collection_summary["success"].append(metric_name)
    elif df is not None:
        collection_summary["skipped"].append(metric_name)
    else:
        collection_summary["failed"].append(metric_name)

# Save updated watermarks
save_watermarks(watermarks)

### 📊 Collection Summary

In [ ]:
print(f"\n{'='*60}")
print(f"  DATA COLLECTION COMPLETE")
print(f"{'='*60}")
print(f"  ✅ Success:  {len(collection_summary['success'])} metrics")
print(f"  ⏭ Skipped:  {len(collection_summary['skipped'])} metrics (no new data)")
print(f"  ❌ Failed:   {len(collection_summary['failed'])} metrics")
print(f"  📂 Storage:  {base_path}")
print(f"{'='*60}")

if collection_summary['failed']:
    print(f"\nFailed metrics: {', '.join(collection_summary['failed'])}")

# Exit value for orchestration
dbutils.notebook.exit(json.dumps({
    "status": "success" if not collection_summary['failed'] else "partial",
    "success_count": len(collection_summary['success']),
    "failed_count": len(collection_summary['failed']),
    "storage_path": base_path,
    "timestamp": now.isoformat(),
}))